# Cloudflare Workers AI — Score Visualizations
Visualizes text, image, and audio scores for Cloudflare across all 56 prompts.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from pathlib import Path

plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = '#f8f9fa'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.4
plt.rcParams['grid.linestyle'] = '--'

CF_COLOR = '#FF8C00'
SCORES_DIR = Path('outputs/scores')
print('Setup complete.')

Setup complete.


## 1. Load Cloudflare Scores

In [ ]:
def load_cf_scores(filename):
    path = SCORES_DIR / filename
    if not path.exists():
        print(f'  [!] {filename} not found — run cloudflare_benchmark.py first')
        return pd.DataFrame()
    df = pd.read_csv(path)
    cf = df[df['model'] == 'cloudflare'].copy()
    if cf.empty:
        print(f'  [!] No cloudflare rows in {filename} — run cloudflare_benchmark.py first')
    else:
        print(f'  Loaded {len(cf)} cloudflare rows from {filename}')
    return cf

text_df  = load_cf_scores('text_scores.csv')
image_df = load_cf_scores('image_scores.csv')
audio_df = load_cf_scores('audio_scores.csv')

---
## 2. TEXT Scores

In [ ]:
if not text_df.empty:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle('Cloudflare — Text Scores per Prompt', fontsize=16, fontweight='bold', y=1.02)

    metrics = [
        ('bertscore',   'BERTScore (0-1)',        'Semantic Similarity'),
        ('rouge',       'ROUGE-L (0-1)',           'Overlap-based Similarity'),
        ('readability', 'Readability (FK/10)',     'Flesch-Kincaid Readability'),
    ]

    for ax, (col, ylabel, title) in zip(axes, metrics):
        vals = text_df[col].values
        x    = text_df['prompt_id'].values if 'prompt_id' in text_df.columns else np.arange(1, len(vals)+1)
        ax.bar(x, vals, color=CF_COLOR, alpha=0.85, width=0.7, edgecolor='white')
        ax.axhline(vals.mean(), color='black', linestyle='--', linewidth=1.5, label=f'Avg = {vals.mean():.3f}')
        ax.set_title(title, fontsize=13, fontweight='bold')
        ax.set_xlabel('Prompt #', fontsize=11)
        ax.set_ylabel(ylabel, fontsize=11)
        ax.legend(fontsize=10)
        ax.spines[['top', 'right']].set_visible(False)

    plt.tight_layout()
    plt.savefig('outputs/graphs/cf_text_per_prompt.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: outputs/graphs/cf_text_per_prompt.png')

In [ ]:
if not text_df.empty:
    avgs = {
        'BERTScore': text_df['bertscore'].mean(),
        'ROUGE-L':   text_df['rouge'].mean(),
        'Readability': text_df['readability'].mean(),
    }

    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.bar(avgs.keys(), avgs.values(), color=CF_COLOR, alpha=0.9, width=0.5, edgecolor='white')
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.005, f'{h:.4f}',
                ha='center', va='bottom', fontsize=12, fontweight='bold')
    ax.set_title('Cloudflare — Average Text Scores', fontsize=15, fontweight='bold')
    ax.set_ylabel('Score', fontsize=12)
    ax.set_ylim(0, max(avgs.values()) * 1.2)
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    plt.savefig('outputs/graphs/cf_text_averages.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: outputs/graphs/cf_text_averages.png')

---
## 3. IMAGE Scores

In [ ]:
if not image_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('Cloudflare — Image Scores per Prompt', fontsize=16, fontweight='bold', y=1.02)

    metrics = [
        ('clip',      'CLIP Score (0-1)',   'Image-Text Alignment (CLIP)'),
        ('aesthetic', 'Aesthetic (0-10)',   'Aesthetic Quality'),
    ]

    for ax, (col, ylabel, title) in zip(axes, metrics):
        vals = image_df[col].values
        x    = image_df['prompt_id'].values if 'prompt_id' in image_df.columns else np.arange(1, len(vals)+1)
        ax.bar(x, vals, color=CF_COLOR, alpha=0.85, width=0.7, edgecolor='white')
        ax.axhline(vals.mean(), color='black', linestyle='--', linewidth=1.5, label=f'Avg = {vals.mean():.3f}')
        ax.set_title(title, fontsize=13, fontweight='bold')
        ax.set_xlabel('Prompt #', fontsize=11)
        ax.set_ylabel(ylabel, fontsize=11)
        ax.legend(fontsize=10)
        ax.spines[['top', 'right']].set_visible(False)

    plt.tight_layout()
    plt.savefig('outputs/graphs/cf_image_per_prompt.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: outputs/graphs/cf_image_per_prompt.png')

In [ ]:
if not image_df.empty:
    avgs = {
        'CLIP':      image_df['clip'].mean(),
        'Aesthetic': image_df['aesthetic'].mean(),
    }

    fig, ax = plt.subplots(figsize=(7, 5))
    bars = ax.bar(avgs.keys(), avgs.values(), color=CF_COLOR, alpha=0.9, width=0.4, edgecolor='white')
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.005, f'{h:.4f}',
                ha='center', va='bottom', fontsize=12, fontweight='bold')
    ax.set_title('Cloudflare — Average Image Scores', fontsize=15, fontweight='bold')
    ax.set_ylabel('Score', fontsize=12)
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    plt.savefig('outputs/graphs/cf_image_averages.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: outputs/graphs/cf_image_averages.png')

---
## 4. AUDIO Scores

In [ ]:
if not audio_df.empty:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle('Cloudflare — Audio Scores per Prompt', fontsize=16, fontweight='bold', y=1.02)

    metrics = [
        ('clap', 'CLAP Score (0-1)',       'Audio-Text Alignment (CLAP)'),
        ('mos',  'MOS (1-5)',              'Mean Opinion Score'),
        ('wer',  'WER (0-1, lower=better)','Word Error Rate'),
    ]

    for ax, (col, ylabel, title) in zip(axes, metrics):
        vals = audio_df[col].values
        x    = audio_df['prompt_id'].values if 'prompt_id' in audio_df.columns else np.arange(1, len(vals)+1)
        color = '#c0392b' if col == 'wer' else CF_COLOR
        ax.bar(x, vals, color=color, alpha=0.85, width=0.7, edgecolor='white')
        ax.axhline(vals.mean(), color='black', linestyle='--', linewidth=1.5, label=f'Avg = {vals.mean():.3f}')
        ax.set_title(title, fontsize=13, fontweight='bold')
        ax.set_xlabel('Prompt #', fontsize=11)
        ax.set_ylabel(ylabel, fontsize=11)
        ax.legend(fontsize=10)
        ax.spines[['top', 'right']].set_visible(False)

    plt.tight_layout()
    plt.savefig('outputs/graphs/cf_audio_per_prompt.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: outputs/graphs/cf_audio_per_prompt.png')

In [ ]:
if not audio_df.empty:
    avgs = {
        'CLAP': audio_df['clap'].mean(),
        'MOS':  audio_df['mos'].mean(),
        'WER':  audio_df['wer'].mean(),
    }
    colors = [CF_COLOR, CF_COLOR, '#c0392b']

    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.bar(avgs.keys(), avgs.values(), color=colors, alpha=0.9, width=0.5, edgecolor='white')
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.005, f'{h:.4f}',
                ha='center', va='bottom', fontsize=12, fontweight='bold')
    ax.set_title('Cloudflare — Average Audio Scores', fontsize=15, fontweight='bold')
    ax.set_ylabel('Score', fontsize=12)
    ax.spines[['top', 'right']].set_visible(False)
    note = mpatches.Patch(color='#c0392b', label='WER: lower is better')
    ax.legend(handles=[note], fontsize=10)
    plt.tight_layout()
    plt.savefig('outputs/graphs/cf_audio_averages.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: outputs/graphs/cf_audio_averages.png')

---
## 5. Summary — All Metrics Overview

In [ ]:
summary = {}

if not text_df.empty:
    summary['BERTScore']   = text_df['bertscore'].mean()
    summary['ROUGE-L']     = text_df['rouge'].mean()
    summary['Readability'] = text_df['readability'].mean()

if not image_df.empty:
    summary['CLIP']      = image_df['clip'].mean()
    summary['Aesthetic'] = image_df['aesthetic'].mean()

if not audio_df.empty:
    summary['CLAP'] = audio_df['clap'].mean()
    summary['MOS']  = audio_df['mos'].mean()
    summary['WER']  = audio_df['wer'].mean()

if summary:
    wer_val = summary.pop('WER', None)

    fig, ax = plt.subplots(figsize=(14, 6))
    keys = list(summary.keys())
    vals = list(summary.values())
    bars = ax.bar(keys, vals, color=CF_COLOR, alpha=0.9, width=0.6, edgecolor='white', label='Higher is better')

    if wer_val is not None:
        ax.bar(['WER'], [wer_val], color='#c0392b', alpha=0.9, width=0.6, edgecolor='white', label='WER: lower is better')
        keys.append('WER')
        vals.append(wer_val)

    for bar in ax.patches:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.003, f'{h:.3f}',
                ha='center', va='bottom', fontsize=10, fontweight='bold')

    ax.set_title('Cloudflare Workers AI — All Metrics Summary (Averages)', fontsize=15, fontweight='bold')
    ax.set_ylabel('Score', fontsize=12)
    ax.set_xlabel('Metric', fontsize=12)
    ax.legend(fontsize=10)
    ax.spines[['top', 'right']].set_visible(False)

    # Section labels
    text_count  = sum(1 for k in ['BERTScore','ROUGE-L','Readability'] if k in summary)
    image_count = sum(1 for k in ['CLIP','Aesthetic'] if k in summary)
    audio_count = sum(1 for k in ['CLAP','MOS','WER'] if k in ['CLAP','MOS'] and k in summary)
    if wer_val is not None:
        audio_count += 1

    plt.tight_layout()
    plt.savefig('outputs/graphs/cf_all_metrics_summary.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: outputs/graphs/cf_all_metrics_summary.png')
    print()
    print('=== CLOUDFLARE FINAL SCORES ===')
    for k, v in {**summary, **(({'WER': wer_val}) if wer_val is not None else {})}.items():
        print(f'  {k:>12}: {v:.4f}')
else:
    print('No Cloudflare score data found. Run cloudflare_benchmark.py first.')